<a href="https://colab.research.google.com/github/AbdulRehman6162/hello_world/blob/master/Profiling_GoogleSheets_Drive_V2_3_1_Fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Restaurant Profiling — Google Maps Collector
## Google Sheets + Google Drive Stable Build

This notebook is a corrected build of the user's V2.3.1 collector.

### Source of truth

**Google Sheets** is the live `Restaurant Master` and `ISB-RWP Target Areas` source.

### Google Drive

Google Drive is used for:

- checkpoint JSON
- scrape log CSV
- raw JSON archive

No local Excel workbook is required for the scraping run.

### Important fixes in this build

- Removes the old Excel-file validation conflict
- Robustly reads Google Sheets with `get_all_values()`
- One shared Google Maps detail extractor for REFRESH and SEARCH
- Review count extraction remains the V2.3.1 aria-label approach
- Standardized/validated operating hours
- Positive-only service signals
- Conservative business status
- Genuine `ChIJ...` Place IDs only
- `0x...:0x...` Maps internal IDs are kept separate
- Existing Place ID is the primary match key
- Different Place IDs can never collide through phone/name fallback
- Automatic Restaurant ID assignment for both new and existing blank-ID rows
- Google Sheet writer updates its in-memory identity map while adding rows
- Existing CRM `Status` is preserved; new rows default to `New Lead`
- Dashboard formulas are repaired directly in Google Sheets
- Per-record Drive checkpoint and scrape log
- No dynamic fallback runner that can hide missing functions

### First run

Use the small validation settings in Cell 4.

Do not switch to a full run until the test output is checked.


In [ ]:
# CELL 1 — Install dependencies

!pip -q install playwright openpyxl pandas nest_asyncio
!playwright install chromium
!apt-get install -y libatk1.0-0
!apt-get install -y libatk-bridge2.0-0
!apt-get install -y libxcomposite1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 18.6 MB/s eta 0:00:00
184.3 MiB [] 0% 0.0s184.3 MiB [] 0% 17.4s184.3 MiB [] 0% 11.9s184.3 MiB [] 1% 5.0s184.3 MiB [] 2% 3.0s184.3 MiB [] 3% 2.4s184.3 MiB [] 4% 2.2s184.3 MiB [] 5% 2.0s184.3 MiB [] 6% 1.8s184.3 MiB [] 6% 2.1s184.3 MiB [] 7% 2.3s184.3 MiB [] 7% 2.4s184.3 MiB [] 8% 2.3s184.3 MiB [] 9% 2.2s184.3 MiB [] 10% 2.0s184.3 MiB [] 11% 1.9s184.3 MiB [] 12% 1.8s184.3 MiB [] 13% 1.8s184.3 MiB [] 14% 1.8s184.3 MiB [] 15% 1.7s184.3 MiB [] 16% 1.6s184.3 MiB [] 17% 1.6s184.3 MiB [] 18% 1.6s184.3 MiB [] 19% 1.6s184.3 MiB [] 20% 1.7s184.3 MiB [] 21% 1.7s184.3 MiB [] 23% 1.6s184.3 MiB [] 24% 1.5s184.3 MiB [] 24% 1.6s184.3 MiB [] 25% 1.6s184.3 MiB [] 27% 1.5s184.3 MiB [] 28% 1.4s184.3 MiB [] 29% 1.4s184.3 MiB [] 31% 1.3s184.3 MiB [] 32% 1.3s184.3 MiB [] 34% 1.2s184.3 MiB [] 36% 1.1s184.3 MiB [] 38% 1.1s184.3 MiB [] 39% 1.0s184.3 MiB [] 40% 1.0s184.3 MiB [] 41% 1.0s184.3 MiB [] 42% 1.0s184.3 MiB [] 43% 1.0s184.3 MiB [] 45% 0.9s184.3 MiB

In [ ]:

# CELL 2 — Imports + Google Drive mount

import os
import re
import json
import random
import asyncio
import traceback
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote, unquote, urlparse

import pandas as pd
import nest_asyncio
import gspread

from google.colab import auth, drive
from google.auth import default

from playwright.async_api import async_playwright

nest_asyncio.apply()

# Mount Drive if needed.
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
else:
    print("Google Drive is already mounted.")

print("Environment loaded.")


Mounted at /content/drive
Environment loaded.


In [ ]:

# CELL 3 — Google Sheets connection

GOOGLE_SHEET_NAME = (
    "Restaurant_Profiling_V2_2.1_data structure GOOGLE SHEET"
)

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

try:
    spreadsheet = gc.open(
        GOOGLE_SHEET_NAME
    )

    print(
        f"Connected to Google Sheet: "
        f"{GOOGLE_SHEET_NAME}"
    )

    print("Worksheets:")
    for ws in spreadsheet.worksheets():
        print(
            f" - {ws.title} "
            f"({ws.id})"
        )

except Exception as exc:
    raise RuntimeError(
        "Could not open the configured Google Sheet. "
        "Check GOOGLE_SHEET_NAME and your Google account permissions."
    ) from exc


Connected to Google Sheet: Restaurant_Profiling_V2_2.1_data structure GOOGLE SHEET
Worksheets:
 - Restaurant Master (1572363453)
 - ISB-RWP Target Areas (1699624687)
 - Collection Methodology (401305918)
 - Pipeline Tracker (98593871)
 - Dashboard (272659734)


In [ ]:

# CELL 4 — Configuration

# ==========================================================
# RUN MODES
# ==========================================================

RUN_REFRESH = True
RUN_SEARCH = True

# ==========================================================
# SAFE FIRST-TEST LIMITS
# ==========================================================

MAX_REFRESH_RECORDS = 10
MAX_SEARCH_QUERIES = 2
MAX_RESULTS_PER_QUERY = 10

# FULL RUN:
# MAX_REFRESH_RECORDS = None
# MAX_SEARCH_QUERIES = None
# MAX_RESULTS_PER_QUERY = 18

# ==========================================================
# SEARCH
# ==========================================================

TARGET_PRIORITY = "P1"

SEARCH_CATEGORIES = [
    "restaurants",
    "cafes",
    "fast food",
    "Pakistani restaurants",
    "BBQ restaurants",
    "Chinese restaurants",
    "pizza restaurants",
    "burger restaurants",
    "coffee shops",
    "bakery",
    "dessert restaurants",
    "fine dining restaurants",
]

# ==========================================================
# BROWSER
# ==========================================================

HEADLESS = True
VIEWPORT = {
    "width": 1440,
    "height": 900,
}

MIN_DELAY = 1.5
MAX_DELAY = 3.5

PAGE_WAIT_MIN = 1.5
PAGE_WAIT_MAX = 3.0

MAX_NAV_RETRIES = 3
RETRY_BASE_DELAY = 5
CAPTCHA_PAUSE = 60

# ==========================================================
# GOOGLE DRIVE OUTPUTS
# ==========================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/"
    "Analytics as a Service/BI Services/"
    "Rest Anchor/Scrapper Output"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

CHECKPOINT_JSON = os.path.join(
    OUTPUT_DIR,
    "v2_3_1_fixed_checkpoint.json"
)

SCRAPE_LOG_CSV = os.path.join(
    OUTPUT_DIR,
    "v2_3_1_fixed_scrape_log.csv"
)

RAW_RESULTS_JSON = os.path.join(
    OUTPUT_DIR,
    "v2_3_1_fixed_raw_results.json"
)

RESET_CHECKPOINT = False

print("Configuration loaded.")
print("Drive output:", OUTPUT_DIR)


Configuration loaded.
Drive output: /content/drive/MyDrive/Analytics as a Service/BI Services/Rest Anchor/Scrapper Output


In [ ]:

# CELL 5 — Google Sheet validation + robust table reader

REQUIRED_SHEETS = [
    "Restaurant Master",
    "ISB-RWP Target Areas",
]

MASTER_COLUMNS = [
    "Restaurant ID",
    "Restaurant Name",
    "Brand Name",
    "Branch #",
    "Format",
    "Cuisine Types",
    "City",
    "Area",
    "Full Address",
    "Latitude",
    "Longitude",
    "Operational Timing",
    "Phone Number",
    "Owner/Manager Contact",
    "Owner/Manager Name",
    "LinkedIn (Y/N)",
    "Dine-In (Y/N)",
    "Takeaway (Y/N)",
    "Delivery (Y/N)",
    "POS Installed (Y/N)",
    "POS System Name",
    "Dedicated BI Team (Y/N)",
    "Website URL",
    "Website Type",
    "Foodpanda Listed (Y/N)",
    "Foodpanda URL",
    "Own Delivery Network (Y/N)",
    "Delivery Partner(s)",
    "Google Business (Y/N)",
    "Google Maps Link",
    "Google Star Rating",
    "Google Review Count",
    "Avg Ticket Size (PKR)",
    "Avg Daily Orders",
    "Data Source",
    "Date Profiled",
    "Status",
    "Notes",
]

V231_COLUMNS = [
    "Place ID",
    "Google Category",
    "Current Open Status",
    "Business Status",
    "Price Level",
    "Data Confidence",
    "Instagram URL",
    "Facebook URL",
]


def normalize_text(value):
    if value is None:
        return ""
    return re.sub(
        r"\s+",
        " ",
        str(value)
    ).strip()


def clean_sheet_values(values):
    """
    Convert get_all_values() output into a DataFrame.
    Keeps row positions aligned with actual Google Sheet rows.
    """
    if not values:
        return pd.DataFrame()

    width = max(
        len(row)
        for row in values
    )

    padded = [
        list(row) + [""] * (
            width - len(row)
        )
        for row in values
    ]

    headers = [
        normalize_text(x)
        for x in padded[0]
    ]

    # If duplicate/blank headers exist, surface a useful warning.
    duplicates = [
        h
        for h in set(headers)
        if h and headers.count(h) > 1
    ]

    if duplicates:
        raise ValueError(
            "Duplicate Google Sheet headers found: "
            + ", ".join(duplicates)
        )

    df = pd.DataFrame(
        padded[1:],
        columns=headers
    )

    # Remove entirely blank data rows.
    if not df.empty:
        df = df.loc[
            ~df.apply(
                lambda row: all(
                    normalize_text(v) == ""
                    for v in row
                ),
                axis=1
            )
        ].reset_index(
            drop=True
        )

    return df


def get_sheet_dataframe(worksheet):
    values = worksheet.get_all_values()
    return clean_sheet_values(values)


def validate_google_sheet():

    titles = [
        ws.title
        for ws in spreadsheet.worksheets()
    ]

    missing = [
        s
        for s in REQUIRED_SHEETS
        if s not in titles
    ]

    if missing:
        raise ValueError(
            "Missing required sheets: "
            + ", ".join(missing)
        )

    master_ws = spreadsheet.worksheet(
        "Restaurant Master"
    )

    master_values = master_ws.get_all_values()

    if not master_values:
        raise ValueError(
            "Restaurant Master is empty."
        )

    headers = [
        normalize_text(x)
        for x in master_values[0]
    ]

    missing_columns = [
        c
        for c in MASTER_COLUMNS
        if c not in headers
    ]

    if missing_columns:
        raise ValueError(
            "Restaurant Master is missing columns: "
            + ", ".join(missing_columns)
        )

    print(
        "Google Sheet validation passed."
    )
    print(
        "Restaurant Master data rows:",
        max(0, len(master_values) - 1)
    )

    print(
        "Restaurant Master columns:",
        len(headers)
    )

    print("\nColumns:")
    for i, h in enumerate(
        headers,
        start=1
    ):
        print(
            f"{i:02d}. {h}"
        )

    return master_ws


master_ws = validate_google_sheet()


Google Sheet validation passed.
Restaurant Master data rows: 0
Restaurant Master columns: 40

Columns:
01. Restaurant ID
02. Restaurant Name
03. Brand Name
04. Branch #
05. Format
06. Cuisine Types
07. City
08. Area
09. Full Address
10. Latitude
11. Longitude
12. Operational Timing
13. Phone Number
14. Owner/Manager Contact
15. Owner/Manager Name
16. LinkedIn (Y/N)
17. Dine-In (Y/N)
18. Takeaway (Y/N)
19. Delivery (Y/N)
20. POS Installed (Y/N)
21. POS System Name
22. Dedicated BI Team (Y/N)
23. Website URL
24. Website Type
25. Foodpanda Listed (Y/N)
26. Foodpanda URL
27. Own Delivery Network (Y/N)
28. Delivery Partner(s)
29. Google Business (Y/N)
30. Google Maps Link
31. Google Star Rating
32. Google Review Count
33. Avg Ticket Size (PKR)
34. Avg Daily Orders
35. Data Source
36. Date Profiled
37. Status
38. Notes
39. Place ID
40. Data Confidence


In [ ]:

# CELL 6 — Parsing helpers

def normalize_name(value):
    value = normalize_text(
        value
    ).lower()

    value = re.sub(
        r"[^a-z0-9\s]",
        " ",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


def normalize_phone(value):
    value = normalize_text(value)

    if not value:
        return ""

    return re.sub(
        r"[^\d+\-\(\)\s]",
        "",
        value
    ).strip()


def phone_key(value):
    return re.sub(
        r"\D",
        "",
        normalize_phone(value)
    )


def first_nonempty(*values):
    for value in values:
        if value not in (
            None,
            "",
            [],
            {}
        ):
            return value
    return None


def parse_count(value):
    if value is None:
        return None

    value = (
        normalize_text(value)
        .upper()
        .replace(",", "")
    )

    if not value:
        return None

    for suffix, multiplier in [
        ("K", 1_000),
        ("M", 1_000_000),
        ("B", 1_000_000_000),
    ]:
        if value.endswith(suffix):
            try:
                return int(
                    float(
                        value[:-1]
                    ) * multiplier
                )
            except Exception:
                return None

    if value.isdigit():
        return int(value)

    return None


def parse_rating(value):
    if not value:
        return None

    text = normalize_text(
        value
    )

    patterns = [
        r"\b([0-5](?:\.\d)?)\s*(?:stars?|/5|rating)\b",
        r"\b([0-5](?:\.\d)?)\b",
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if match:
            try:
                number = float(
                    match.group(1)
                )

                if 0 <= number <= 5:
                    return number

            except Exception:
                pass

    return None


def extract_coordinates(url):

    if not url:
        return None, None

    patterns = [
        r"/@(-?\d+\.\d+),(-?\d+\.\d+)",
        r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)",
        r"@(-?\d+(?:\.\d+)?),(-?\d+(?:\.\d+)?)",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            url
        )

        if match:
            try:
                return (
                    float(match.group(1)),
                    float(match.group(2))
                )
            except Exception:
                pass

    return None, None


def parse_day_name(text):

    mapping = {
        "mon": "Mon",
        "monday": "Mon",
        "tue": "Tue",
        "tues": "Tue",
        "tuesday": "Tue",
        "wed": "Wed",
        "wednesday": "Wed",
        "thu": "Thu",
        "thur": "Thu",
        "thurs": "Thu",
        "thursday": "Thu",
        "fri": "Fri",
        "friday": "Fri",
        "sat": "Sat",
        "saturday": "Sat",
        "sun": "Sun",
        "sunday": "Sun",
    }

    return mapping.get(
        normalize_text(
            text
        ).lower()
    )

print("Parsing helpers loaded.")


Parsing helpers loaded.


In [ ]:

# CELL 7 — Identity + classification helpers

def extract_place_id_from_url(url):
    """
    Extract genuine ChIJ-style Place IDs.
    Do NOT treat 0x...:0x... grid/internal IDs as Place IDs.
    """
    if not url:
        return ""

    decoded = unquote(url)

    patterns = [
        r"!1s(ChIJ[A-Za-z0-9_\-]+)",
        r"!19s(ChIJ[A-Za-z0-9_\-]+)",
        r"[?&]query_place_id=(ChIJ[A-Za-z0-9_\-]+)",
        r"place_id=(ChIJ[A-Za-z0-9_\-]+)",
    ]

    for pattern in patterns:

        match = re.search(
            pattern,
            decoded
        )

        if match:
            return match.group(1)

    return ""


def extract_maps_internal_id(url):

    if not url:
        return ""

    decoded = unquote(
        url
    )

    match = re.search(
        r"!1s(0x[0-9a-fA-F]+:0x[0-9a-fA-F]+)",
        decoded
    )

    if match:
        return match.group(1)

    return ""


def make_identity_key(record):

    place_id = normalize_text(
        record.get("place_id")
    )

    if place_id:
        return (
            "place_id:"
            + place_id
        )

    maps_url = normalize_text(
        record.get("maps_url")
    )

    if maps_url:
        return (
            "maps:"
            + maps_url.split("?")[0].rstrip("/")
        )

    phone = phone_key(
        record.get("phone")
    )

    if phone:
        return (
            "phone:"
            + phone
        )

    name = normalize_name(
        record.get("name")
    )

    address = normalize_name(
        record.get("address")
    )

    if name and address:
        return (
            f"name_address:{name}|{address}"
        )

    lat = record.get("latitude")
    lon = record.get("longitude")

    if (
        name
        and lat is not None
        and lon is not None
    ):
        return (
            f"name_coord:{name}|"
            f"{round(float(lat), 5)}|"
            f"{round(float(lon), 5)}"
        )

    return (
        "name:"
        + name
    )


def infer_format(
    category,
    name
):

    cat = normalize_text(
        category
    ).lower()

    name_text = normalize_text(
        name
    ).lower()

    if cat:

        if any(
            x in cat
            for x in [
                "coffee",
                "cafe",
                "café"
            ]
        ):
            return "Cafe"

        if any(
            x in cat
            for x in [
                "bakery",
                "bakers"
            ]
        ):
            return "Bakery"

        if any(
            x in cat
            for x in [
                "ice cream",
                "dessert",
                "sweets",
                "mithai"
            ]
        ):
            return "Dessert / Sweets"

        if any(
            x in cat
            for x in [
                "fast food",
                "burger",
                "pizza",
                "qsr",
                "fried chicken"
            ]
        ):
            return "QSR"

        if any(
            x in cat
            for x in [
                "fine dining",
                "steakhouse",
                "steak house"
            ]
        ):
            return "Full-Service Restaurant"

        if any(
            x in cat
            for x in [
                "dhaba",
                "dhabba"
            ]
        ):
            return "Dhaba"

        if "food court" in cat or "food truck" in cat:
            return "Food Court / Truck"

        if "restaurant" in cat:
            return "Restaurant"

    if (
        "coffee" in name_text
        or "cafe" in name_text
        or "café" in name_text
    ):
        return "Cafe"

    if (
        "bakery" in name_text
        or "bakers" in name_text
    ):
        return "Bakery"

    if any(
        x in name_text
        for x in [
            "burger",
            "pizza",
            "fried chicken"
        ]
    ):
        return "QSR"

    return "Restaurant"


KNOWN_BRANDS = [
    "Monal",
    "Tuscany Courtyard",
    "Burning Brownie",
    "JEETO",
    "Savour Foods",
    "Jalal Sons",
    "Tehzeeb Bakers",
    "Gourmet",
    "Student Biryani",
    "Bundu Khan",
    "Bar.B.Q Tonight",
    "BBQ Tonight",
    "Howdy",
    "KFC",
    "McDonald's",
    "Pizza Hut",
    "Domino's",
    "Hardee's",
    "Subway",
    "Nando's",
    "OPTP",
    "Broadway Pizza",
    "14th Street Pizza",
    "Kababjees",
    "Kolachi",
    "Café Aylanto",
    "Chaaye Khana",
    "Mocca Coffee",
    "Gloria Jean's",
    "Dunkin' Donuts",
    "Baskin Robbins",
    "Cold Stone",
    "Layers Bakeshop",
    "Pie in the Sky",
    "Roasters",
    "Juice Junction",
    "Des Pardes",
    "Haveli",
    "Second Cup",
    "Zus Coffee",
    "KAF Coffee",
    "Eggspectation",
    "Caffé Praha",
]


def conservative_brand(name):

    name = normalize_text(
        name
    )

    if not name:
        return ""

    lower = name.lower()

    for brand in KNOWN_BRANDS:

        if brand.lower() in lower:
            return brand

    # IMPORTANT:
    # Split only on spaced separators so F-6 is not split.
    value = re.split(
        r"\s+-\s+|\s*\|\s*|\s*,\s+",
        name
    )[0].strip()

    value = re.sub(
        r"\b(DHA|Blue Area|Saddar|Bahria Town|"
        r"Johar Town|Gulberg)\b.*$",
        "",
        value,
        flags=re.IGNORECASE
    ).strip(
        " -|,"
    )

    return value


def classify_website_type(url):

    if not url:
        return ""

    lower = url.lower()

    mapping = [
        ("foodpanda", "Foodpanda"),
        ("eat.cheetay", "Cheetay"),
        ("careem.com", "Careem"),
        ("bykea.com", "Bykea"),
        ("instagram.com", "Instagram"),
        ("facebook.com", "Facebook"),
        ("fb.com", "Facebook"),
        ("tiktok.com", "TikTok"),
        ("linktree", "Linktree"),
        ("linktr.ee", "Linktree"),
    ]

    for key, label in mapping:

        if key in lower:
            return label

    return "Own Website"


def infer_confidence(record):

    problems = []

    if not record.get("name"):
        problems.append(
            "name missing"
        )

    if not record.get("maps_url"):
        problems.append(
            "Maps URL missing"
        )

    if not record.get("place_id"):
        problems.append(
            "Place ID missing"
        )

    if record.get("rating") is None:
        problems.append(
            "rating missing"
        )

    if record.get("review_count") is None:
        problems.append(
            "review count missing"
        )

    if not record.get("address"):
        problems.append(
            "address missing"
        )

    if not record.get("phone"):
        problems.append(
            "phone missing"
        )

    if record.get(
        "review_source"
    ) == "body-fallback":
        problems.append(
            "review count from fallback"
        )

    if record.get("error"):
        problems.append(
            "extraction error"
        )

    if problems:
        return (
            "REVIEW: "
            + "; ".join(problems)
        )

    return "OK"

print("Identity/classification helpers loaded.")


Identity/classification helpers loaded.


In [ ]:

# CELL 8 — Browser/navigation helpers

async def detect_captcha(page):

    try:

        body = await page.locator(
            "body"
        ).inner_text(
            timeout=5_000
        )

        lower = body.lower()

        signals = [
            "unusual traffic",
            "captcha",
            "are you a robot",
            "automated queries",
            "please show you're not a robot",
            "please show you’re not a robot",
            "sorry, we can't verify",
            "sorry, we can’t verify",
        ]

        return any(
            s in lower
            for s in signals
        )

    except Exception:
        return False


async def accept_google_consent(page):

    for text in [
        "Accept all",
        "I agree",
        "Accept"
    ]:

        try:

            locator = page.get_by_role(
                "button",
                name=re.compile(
                    rf"^{re.escape(text)}$",
                    re.IGNORECASE
                )
            )

            if await locator.count():

                await locator.first.click(
                    timeout=2_000
                )

                await page.wait_for_timeout(
                    800
                )

                return True

        except Exception:
            pass

    return False


async def safe_goto(
    page,
    url,
    wait_ms=None
):

    last_error = None

    for attempt in range(
        MAX_NAV_RETRIES
    ):

        try:

            await page.goto(
                url,
                wait_until="domcontentloaded",
                timeout=60_000
            )

            await accept_google_consent(
                page
            )

            if await detect_captcha(
                page
            ):

                delay = (
                    CAPTCHA_PAUSE
                    * (attempt + 1)
                )

                print(
                    f"  ⚠ CAPTCHA detected. "
                    f"Pausing {delay}s."
                )

                await asyncio.sleep(
                    delay
                )

                continue

            if wait_ms is None:
                wait_ms = random.randint(
                    int(PAGE_WAIT_MIN * 1000),
                    int(PAGE_WAIT_MAX * 1000)
                )

            await page.wait_for_timeout(
                wait_ms
            )

            return

        except Exception as exc:

            last_error = exc

            delay = (
                RETRY_BASE_DELAY
                * (2 ** attempt)
            )

            print(
                f"  ⚠ Navigation failed "
                f"(attempt {attempt + 1}/"
                f"{MAX_NAV_RETRIES}). "
                f"Retrying in {delay}s."
            )

            await asyncio.sleep(
                delay
            )

    raise last_error or RuntimeError(
        "Navigation failed."
    )

print("Browser helpers loaded.")


Browser helpers loaded.


In [ ]:

# CELL 9 — Review + rating extraction

async def extract_review_count(page):

    candidates = []

    # 1. aria-labels
    try:

        values = await page.locator(
            "[aria-label]"
        ).evaluate_all(
            """
            els => els
              .map(e => e.getAttribute('aria-label'))
              .filter(Boolean)
              .filter(v => /review|rating|stars?/i.test(v))
              .slice(0, 250)
            """
        )

        candidates.extend(
            [
                ("aria-label", value)
                for value in values
            ]
        )

    except Exception:
        pass

    # 2. focused review/rating elements
    selectors = [
        'div[jsaction*="moreReviews"]',
        'div[jsaction*="reviews"]',
        'button[aria-label*="review" i]',
        'button[aria-label*="rating" i]',
        '[role="img"][aria-label*="star" i]',
    ]

    for selector in selectors:

        try:

            elements = page.locator(
                selector
            )

            count = await elements.count()

            for i in range(
                min(count, 30)
            ):

                try:

                    el = elements.nth(i)

                    aria = await el.get_attribute(
                        "aria-label"
                    )

                    text = await el.inner_text()

                    if aria:
                        candidates.append(
                            (
                                selector + ":aria",
                                aria
                            )
                        )

                    if text:
                        candidates.append(
                            (
                                selector + ":text",
                                text
                            )
                        )

                except Exception:
                    pass

        except Exception:
            pass

    patterns = [
        r"([\d,.]+[KMB]?)\s+reviews?",
        r"([\d,.]+[KMB]?)\s+ratings?",
        r"\(([\d,.]+[KMB]?)\)",
    ]

    for source, text in candidates:

        text = normalize_text(
            text
        )

        for pattern in patterns:

            match = re.search(
                pattern,
                text,
                re.IGNORECASE
            )

            if match:

                count = parse_count(
                    match.group(1)
                )

                if count is not None:

                    return {
                        "value": count,
                        "source": (
                            "aria-label"
                            if "aria-label" in source
                            else source
                        ),
                        "raw": text,
                    }

    # Tight fallback
    try:

        body = await page.locator(
            "body"
        ).inner_text(
            timeout=10_000
        )

        for line in body.splitlines():

            line = normalize_text(
                line
            )

            if not re.search(
                r"\breviews?\b",
                line,
                re.IGNORECASE
            ):
                continue

            match = re.search(
                r"([\d,.]+[KMB]?)\s+reviews?",
                line,
                re.IGNORECASE
            )

            if match:

                count = parse_count(
                    match.group(1)
                )

                if count is not None:

                    return {
                        "value": count,
                        "source": "body-fallback",
                        "raw": line,
                    }

    except Exception:
        pass

    return {
        "value": None,
        "source": "not-found",
        "raw": "",
    }


async def extract_rating(page):

    candidates = []

    selectors = [
        '[aria-label*="stars" i]',
        '[aria-label*="star" i]',
        '[aria-label*="rating" i]',
        'div[jsaction*="moreReviews"]',
    ]

    for selector in selectors:

        try:

            elements = page.locator(
                selector
            )

            count = await elements.count()

            for i in range(
                min(count, 25)
            ):

                try:

                    el = elements.nth(i)

                    aria = await el.get_attribute(
                        "aria-label"
                    )

                    text = await el.inner_text()

                    if aria:
                        candidates.append(
                            aria
                        )

                    if text:
                        candidates.append(
                            text
                        )

                except Exception:
                    pass

        except Exception:
            pass

    for text in candidates:

        rating = parse_rating(
            text
        )

        if rating is not None:
            return rating

    return None

print("Review/rating extractors loaded.")


Review/rating extractors loaded.


In [ ]:

# CELL 10 — Operating hours extraction

TIME_TOKEN_RE = re.compile(
    r"\b(?:1[0-2]|0?[1-9])"
    r"(?::[0-5]\d)?\s*"
    r"(?:AM|PM|am|pm)"
)

VALID_SPECIAL_HOURS = {
    "closed",
    "open 24 hours",
    "24 hours",
    "open all day",
}


def extract_time_range(text):

    if not text:
        return ""

    text = normalize_text(
        text
    )

    if text.lower() in VALID_SPECIAL_HOURS:
        return text

    pattern = (
        r"(\d{1,2}(?::\d{2})?\s*[AP]M)"
        r"\s*(?:to|–|-|—)"
        r"\s*(\d{1,2}(?::\d{2})?\s*[AP]M)"
    )

    match = re.search(
        pattern,
        text,
        re.IGNORECASE
    )

    if match:

        return (
            f"{match.group(1).upper()}–"
            f"{match.group(2).upper()}"
        )

    matches = TIME_TOKEN_RE.findall(
        text
    )

    if len(matches) >= 2:

        return (
            f"{matches[0].upper()}–"
            f"{matches[1].upper()}"
        )

    if len(matches) == 1:
        return matches[0].upper()

    return ""


def is_valid_hour_value(value):

    if not value:
        return False

    value = normalize_text(
        value
    )

    if value.lower() in VALID_SPECIAL_HOURS:
        return True

    return bool(
        TIME_TOKEN_RE.search(
            value
        )
    )


async def extract_hours(page):

    day_map = {
        "Monday": "Mon",
        "Tuesday": "Tue",
        "Wednesday": "Wed",
        "Thursday": "Thu",
        "Friday": "Fri",
        "Saturday": "Sat",
        "Sunday": "Sun",
    }

    found = {}
    candidates = []

    # aria-label
    try:

        values = await page.locator(
            "[aria-label]"
        ).evaluate_all(
            """
            els => els
              .map(e => e.getAttribute('aria-label'))
              .filter(Boolean)
              .filter(v => /Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday/i.test(v))
              .slice(0, 200)
            """
        )

        candidates.extend(
            values
        )

    except Exception:
        pass

    # data-tooltip
    try:

        values = await page.locator(
            "[data-tooltip]"
        ).evaluate_all(
            """
            els => els
              .map(e => e.getAttribute('data-tooltip'))
              .filter(Boolean)
              .filter(v => /Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday/i.test(v))
              .slice(0, 200)
            """
        )

        candidates.extend(
            values
        )

    except Exception:
        pass

    for raw in candidates:

        text = normalize_text(
            raw
        )

        match = re.search(
            r"^(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)"
            r"\s*[:,\-]?\s*(.*)$",
            text,
            re.IGNORECASE
        )

        if not match:
            continue

        day_name = match.group(1).title()
        day = day_map.get(
            day_name
        )

        rest = normalize_text(
            match.group(2)
        )

        if not day:
            continue

        value = extract_time_range(
            rest
        )

        if (
            value
            and is_valid_hour_value(
                value
            )
        ):
            found[day] = value

    # body fallback
    if not found:

        try:

            body = await page.locator(
                "body"
            ).inner_text()

            for raw in body.splitlines():

                text = normalize_text(
                    raw
                )

                match = re.search(
                    r"^(Monday|Tuesday|Wednesday|Thursday|Friday|Saturday|Sunday)"
                    r"\s*[:,\-]?\s*(.*)$",
                    text,
                    re.IGNORECASE
                )

                if not match:
                    continue

                day_name = match.group(1).title()
                day = day_map.get(
                    day_name
                )

                rest = normalize_text(
                    match.group(2)
                )

                value = extract_time_range(
                    rest
                )

                if (
                    day
                    and value
                    and is_valid_hour_value(
                        value
                    )
                ):
                    found[day] = value

        except Exception:
            pass

    ordered = []

    for day in [
        "Mon",
        "Tue",
        "Wed",
        "Thu",
        "Fri",
        "Sat",
        "Sun",
    ]:

        if day in found:
            ordered.append(
                f"{day}: {found[day]}"
            )

    return " | ".join(
        ordered
    )


def compress_hours(hours):

    if not hours:
        return ""

    parsed = []

    for part in hours.split("|"):

        part = normalize_text(
            part
        )

        match = re.match(
            r"^(Mon|Tue|Wed|Thu|Fri|Sat|Sun)"
            r"\s*:\s*(.+)$",
            part
        )

        if not match:
            continue

        day = match.group(1)
        value = normalize_text(
            match.group(2)
        )

        if not is_valid_hour_value(
            value
        ):
            continue

        parsed.append(
            (day, value)
        )

    if not parsed:
        return ""

    if len(parsed) < 7:

        return " | ".join(
            f"{d}: {v}"
            for d, v in parsed
        )

    groups = []

    start_day = parsed[0][0]
    current_value = parsed[0][1]

    for i in range(
        1,
        len(parsed)
    ):

        if parsed[i][1] == current_value:
            continue

        groups.append(
            (
                start_day,
                parsed[i - 1][0],
                current_value
            )
        )

        start_day = parsed[i][0]
        current_value = parsed[i][1]

    groups.append(
        (
            start_day,
            parsed[-1][0],
            current_value
        )
    )

    parts = []

    for start, end, value in groups:

        if start == end:
            parts.append(
                f"{start}: {value}"
            )
        else:
            parts.append(
                f"{start}–{end}: {value}"
            )

    return " | ".join(
        parts
    )

print("Operating-hours extractor loaded.")


Operating-hours extractor loaded.


In [ ]:

# CELL 11 — Service signals + status

def detect_service_token(text):

    text = normalize_text(
        text
    ).lower()

    result = {
        "dine_in": "",
        "takeaway": "",
        "delivery": "",
    }

    if re.search(
        r"\bdine[\s-]?in\b",
        text
    ):
        result["dine_in"] = "Y"

    if re.search(
        r"\btake[\s-]?out\b|\btakeaway\b",
        text
    ):
        result["takeaway"] = "Y"

    if re.search(
        r"\bdelivery\b",
        text
    ):
        result["delivery"] = "Y"

    return result


async def extract_service_signals(page):

    result = {
        "dine_in": "",
        "takeaway": "",
        "delivery": "",
    }

    candidates = []

    for attr in [
        "aria-label",
        "data-tooltip",
        "title",
    ]:

        try:

            values = await page.locator(
                f"[{attr}]"
            ).evaluate_all(
                f"""
                els => els
                  .map(e => e.getAttribute('{attr}'))
                  .filter(Boolean)
                  .filter(v => /dine|take.?out|takeaway|delivery/i.test(v))
                  .slice(0, 250)
                """
            )

            candidates.extend(
                values
            )

        except Exception:
            pass

    # Explicit service UI/button text only.
    try:

        buttons = page.locator(
            "button"
        )

        count = await buttons.count()

        for i in range(
            min(count, 200)
        ):

            try:

                el = buttons.nth(i)

                aria = await el.get_attribute(
                    "aria-label"
                )

                text = await el.inner_text()

                if aria:
                    candidates.append(
                        aria
                    )

                if text:
                    candidates.append(
                        text
                    )

            except Exception:
                pass

    except Exception:
        pass

    for candidate in candidates:

        detected = detect_service_token(
            candidate
        )

        for key, value in detected.items():

            if value == "Y":
                result[key] = "Y"

    # Intentionally NO generic body fallback.
    return result


async def extract_business_status(page):

    current_open_status = "Unknown"
    business_status = "Unknown"

    try:

        body = await page.locator(
            "body"
        ).inner_text()

        text = normalize_text(
            body[:10_000]
        )

        lower = text.lower()

        if "permanently closed" in lower:

            business_status = (
                "Permanently Closed"
            )

        elif "temporarily closed" in lower:

            business_status = (
                "Temporarily Closed"
            )

        else:

            # We can establish current operational
            # state separately from permanent status.
            if re.search(
                r"\bOpen\s*·|\bOpen now\b",
                text,
                re.IGNORECASE
            ):
                current_open_status = "Open"

            elif re.search(
                r"\bClosed\s*·|\bOpens\s",
                text,
                re.IGNORECASE
            ):
                current_open_status = "Closed"

            # Only infer operational when there is
            # clear active listing evidence.
            active_signals = [
                "open now",
                "closes at",
                "opens at",
                "open 24 hours",
            ]

            if (
                any(
                    sig in lower
                    for sig in active_signals
                )
                or re.search(
                    r"\d+\s+reviews?",
                    lower
                )
            ):
                business_status = "Operational"

    except Exception:
        pass

    return {
        "current_open_status": current_open_status,
        "business_status": business_status,
    }


async def extract_price_level(page):

    try:

        values = await page.locator(
            "[aria-label]"
        ).evaluate_all(
            """
            els => els
              .map(e => e.getAttribute('aria-label'))
              .filter(Boolean)
              .filter(v => /price|price range|\\$\\$|moderately|inexpensive|expensive/i.test(v))
              .slice(0, 100)
            """
        )

        for value in values:

            match = re.search(
                r"(\${1,5})",
                value
            )

            if match:
                return match.group(1)

    except Exception:
        pass

    return ""


async def extract_social_links(page):

    result = {
        "instagram": "",
        "facebook": "",
        "foodpanda_url": "",
    }

    try:

        hrefs = await page.locator(
            "a[href]"
        ).evaluate_all(
            """
            els => els
              .map(e => e.getAttribute('href'))
              .filter(Boolean)
              .slice(0, 500)
            """
        )

        for href in hrefs:

            lower = href.lower()

            if (
                "instagram.com/"
                in lower
                and not result["instagram"]
            ):
                result["instagram"] = href

            elif (
                (
                    "facebook.com/"
                    in lower
                    or "fb.com/"
                    in lower
                )
                and not result["facebook"]
            ):
                result["facebook"] = href

            elif (
                "foodpanda"
                in lower
                and not result["foodpanda_url"]
            ):
                result["foodpanda_url"] = href

    except Exception:
        pass

    return result

print("Service/status/social extractors loaded.")


Service/status/social extractors loaded.


In [ ]:

# CELL 12 — Core business extractors

async def extract_phone(page):

    selectors = [
        'button[data-item-id^="phone:tel:"]',
        'button[aria-label*="Phone" i]',
        'a[href^="tel:"]',
    ]

    for selector in selectors:

        try:

            locator = page.locator(
                selector
            ).first

            if await locator.count():

                aria = await locator.get_attribute(
                    "aria-label"
                )

                text = await locator.inner_text()

                value = first_nonempty(
                    aria,
                    text
                )

                if value:

                    value = re.sub(
                        r"^Phone\s*:\s*",
                        "",
                        value,
                        flags=re.IGNORECASE
                    )

                    value = normalize_phone(
                        value
                    )

                    if value:
                        return value

        except Exception:
            pass

    return ""


async def extract_address(page):

    selectors = [
        'button[data-item-id="address"]',
        'button[aria-label^="Address:" i]',
        'button[aria-label*="Address" i]',
    ]

    for selector in selectors:

        try:

            locator = page.locator(
                selector
            ).first

            if await locator.count():

                aria = await locator.get_attribute(
                    "aria-label"
                )

                text = await locator.inner_text()

                value = first_nonempty(
                    aria,
                    text
                )

                if value:

                    value = re.sub(
                        r"^Address\s*:\s*",
                        "",
                        value,
                        flags=re.IGNORECASE
                    )

                    return normalize_text(
                        value
                    )

        except Exception:
            pass

    return ""


async def extract_website(page):

    selectors = [
        'a[data-item-id="authority"]',
        'a[aria-label*="Website" i]',
    ]

    for selector in selectors:

        try:

            locator = page.locator(
                selector
            ).first

            if await locator.count():

                href = await locator.get_attribute(
                    "href"
                )

                if href:
                    return href

        except Exception:
            pass

    return ""


async def extract_google_category(
    page,
    name
):

    selectors = [
        'button[jsaction*="category"]',
        'a[href*="/maps/place/"][aria-label]',
    ]

    for selector in selectors:

        try:

            elements = page.locator(
                selector
            )

            count = await elements.count()

            for i in range(
                min(count, 20)
            ):

                try:

                    text = normalize_text(
                        await elements.nth(i).inner_text()
                    )

                    if (
                        text
                        and text.lower()
                        != normalize_text(
                            name
                        ).lower()
                        and 2 <= len(text) <= 100
                    ):
                        return text

                except Exception:
                    pass

        except Exception:
            pass

    return ""

print("Core extractors loaded.")


Core extractors loaded.


In [ ]:

# CELL 13 — SINGLE shared place-detail extractor

async def extract_place_details(
    page,
    *,
    url,
    city="",
    area="",
    search_category="",
    mode="SEARCH"
):

    record = {
        "mode": mode,
        "city": city,
        "area": area,
        "search_category": search_category,

        "name": "",
        "category": "",
        "address": "",
        "phone": "",
        "website": "",
        "website_type": "",

        "rating": None,
        "review_count": None,
        "review_source": "not-found",
        "review_raw": "",

        "hours": "",
        "latitude": None,
        "longitude": None,

        "dine_in": "",
        "takeaway": "",
        "delivery": "",

        "current_open_status": "Unknown",
        "business_status": "Unknown",
        "price_level": "",

        "maps_url": "",
        "place_id": "",
        "maps_internal_id": "",

        "format": "",
        "rough_brand": "",

        "data_confidence": "",
        "error": "",

        "instagram_url": "",
        "facebook_url": "",
        "foodpanda_url": "",
        "foodpanda_listed": "",
    }

    try:

        await safe_goto(
            page,
            url
        )

        record["maps_url"] = page.url

        # Name
        try:

            h1 = page.locator(
                "h1"
            ).first

            if await h1.count():

                record["name"] = normalize_text(
                    await h1.inner_text()
                )

        except Exception:
            pass

        if not record["name"]:

            record["error"] = (
                "Restaurant name not found"
            )

            record["data_confidence"] = (
                "REVIEW: name missing"
            )

            return record

        # Core
        record["rating"] = (
            await extract_rating(page)
        )

        review = await extract_review_count(
            page
        )

        record["review_count"] = (
            review["value"]
        )

        record["review_source"] = (
            review["source"]
        )

        record["review_raw"] = (
            review["raw"]
        )

        record["address"] = (
            await extract_address(page)
        )

        record["phone"] = (
            await extract_phone(page)
        )

        record["website"] = (
            await extract_website(page)
        )

        record["website_type"] = (
            classify_website_type(
                record["website"]
            )
        )

        record["category"] = (
            await extract_google_category(
                page,
                record["name"]
            )
        )

        raw_hours = await extract_hours(
            page
        )

        record["hours"] = compress_hours(
            raw_hours
        )

        record["latitude"], record["longitude"] = (
            extract_coordinates(
                record["maps_url"]
            )
        )

        record["place_id"] = (
            extract_place_id_from_url(
                record["maps_url"]
            )
        )

        record["maps_internal_id"] = (
            extract_maps_internal_id(
                record["maps_url"]
            )
        )

        # Services
        services = (
            await extract_service_signals(
                page
            )
        )

        record["dine_in"] = (
            services["dine_in"]
        )

        record["takeaway"] = (
            services["takeaway"]
        )

        record["delivery"] = (
            services["delivery"]
        )

        # Status
        status = (
            await extract_business_status(
                page
            )
        )

        record["current_open_status"] = (
            status["current_open_status"]
        )

        record["business_status"] = (
            status["business_status"]
        )

        record["price_level"] = (
            await extract_price_level(
                page
            )
        )

        # Preliminary external links
        social = (
            await extract_social_links(
                page
            )
        )

        record["instagram_url"] = (
            social["instagram"]
        )

        record["facebook_url"] = (
            social["facebook"]
        )

        record["foodpanda_url"] = (
            social["foodpanda_url"]
        )

        record["foodpanda_listed"] = (
            "Y"
            if record["foodpanda_url"]
            else ""
        )

        # Derived fields
        record["format"] = infer_format(
            record["category"],
            record["name"]
        )

        record["rough_brand"] = (
            conservative_brand(
                record["name"]
            )
        )

        record["data_confidence"] = (
            infer_confidence(
                record
            )
        )

        return record

    except Exception as exc:

        record["error"] = (
            f"{type(exc).__name__}: "
            f"{str(exc)[:500]}"
        )

        record["data_confidence"] = (
            "REVIEW: extraction exception"
        )

        return record

print("Shared place-detail extractor loaded.")


Shared place-detail extractor loaded.


In [ ]:

# CELL 14 — Search collector

async def collect_search_results(
    page,
    query,
    max_results=30
):

    search_url = (
        "https://www.google.com/maps/search/"
        + quote(query)
    )

    print(
        f"\nSEARCH: {query}"
    )

    try:

        await safe_goto(
            page,
            search_url,
            wait_ms=3_000
        )

    except Exception as exc:

        print(
            "Search navigation error:",
            exc
        )

        return []

    results = {}

    try:

        await page.wait_for_selector(
            'div[role="feed"]',
            timeout=15_000
        )

    except Exception:
        pass

    previous_count = 0
    stagnant_rounds = 0

    for round_no in range(20):

        try:

            anchors = page.locator(
                'a[href*="/maps/place/"]'
            )

            count = await anchors.count()

            for i in range(count):

                try:

                    a = anchors.nth(i)

                    href = await a.get_attribute(
                        "href"
                    )

                    aria = await a.get_attribute(
                        "aria-label"
                    )

                    text = normalize_text(
                        await a.inner_text()
                    )

                    if (
                        not href
                        or "/maps/place/"
                        not in href
                    ):
                        continue

                    results[href] = {
                        "url": href,
                        "label": normalize_text(
                            aria or text
                        ),
                    }

                except Exception:
                    pass

        except Exception:
            pass

        current_count = len(
            results
        )

        print(
            f"  scroll {round_no + 1:02d}: "
            f"{current_count} URLs"
        )

        if current_count >= max_results:
            break

        if (
            current_count
            == previous_count
        ):
            stagnant_rounds += 1
        else:
            stagnant_rounds = 0

        if stagnant_rounds >= 4:
            break

        previous_count = current_count

        try:

            feed = page.locator(
                'div[role="feed"]'
            )

            if await feed.count():

                await feed.evaluate(
                    "(el) => el.scrollTop = el.scrollHeight"
                )

            else:

                await page.mouse.wheel(
                    0,
                    5000
                )

        except Exception:

            try:
                await page.mouse.wheel(
                    0,
                    5000
                )
            except Exception:
                pass

        await page.wait_for_timeout(
            random.randint(
                1200,
                2500
            )
        )

    return list(
        results.values()
    )[:max_results]

print("Search collector loaded.")


Search collector loaded.


In [ ]:

# CELL 15 — Target-area query generation from Google Sheets

def generate_queries(
    priority=TARGET_PRIORITY
):

    ws = spreadsheet.worksheet(
        "ISB-RWP Target Areas"
    )

    df = get_sheet_dataframe(
        ws
    )

    if df.empty:
        raise ValueError(
            "ISB-RWP Target Areas contains no data rows."
        )

    required = [
        "City",
        "Area/Sector",
        "Priority",
    ]

    missing = [
        c
        for c in required
        if c not in df.columns
    ]

    if missing:
        raise ValueError(
            "Target area sheet is missing: "
            + ", ".join(missing)
        )

    selected = df[
        df["Priority"].astype(str)
        .str.upper()
        .str.strip()
        == str(priority)
        .upper()
        .strip()
    ]

    queries = []

    for _, row in selected.iterrows():

        city = normalize_text(
            row["City"]
        )

        area_text = normalize_text(
            row["Area/Sector"]
        )

        if not city or not area_text:
            continue

        areas = [
            normalize_text(x)
            for x in re.split(
                r"[,;/|]",
                area_text
            )
            if normalize_text(x)
        ]

        for area in areas:

            for category in SEARCH_CATEGORIES:

                queries.append({
                    "city": city,
                    "area": area,
                    "category": category,
                    "priority": priority,
                    "query": (
                        f"{category} in "
                        f"{area} {city}"
                    ),
                })

    return queries


queries = generate_queries()

print(
    "Generated queries:",
    len(queries)
)

for item in queries[:20]:
    print(
        "-",
        item["query"]
    )


Generated queries: 180
- restaurants in F-6 Islamabad
- cafes in F-6 Islamabad
- fast food in F-6 Islamabad
- Pakistani restaurants in F-6 Islamabad
- BBQ restaurants in F-6 Islamabad
- Chinese restaurants in F-6 Islamabad
- pizza restaurants in F-6 Islamabad
- burger restaurants in F-6 Islamabad
- coffee shops in F-6 Islamabad
- bakery in F-6 Islamabad
- dessert restaurants in F-6 Islamabad
- fine dining restaurants in F-6 Islamabad
- restaurants in F-7 Islamabad
- cafes in F-7 Islamabad
- fast food in F-7 Islamabad
- Pakistani restaurants in F-7 Islamabad
- BBQ restaurants in F-7 Islamabad
- Chinese restaurants in F-7 Islamabad
- pizza restaurants in F-7 Islamabad
- burger restaurants in F-7 Islamabad


In [ ]:

# CELL 16 — Existing Restaurant Master reader from Google Sheets

def read_existing_master():

    ws = spreadsheet.worksheet(
        "Restaurant Master"
    )

    values = ws.get_all_values()

    if not values:
        return []

    df = clean_sheet_values(
        values
    )

    if df.empty:
        return []

    records = []

    # After removing blank rows, we cannot use
    # DataFrame index as an actual sheet row if blank
    # rows existed in the middle. Instead, scan the
    # original values to build actual row numbers.
    headers = [
        normalize_text(x)
        for x in values[0]
    ]

    for actual_row, raw_row in enumerate(
        values[1:],
        start=2
    ):

        padded = list(raw_row) + [
            ""
        ] * (
            len(headers)
            - len(raw_row)
        )

        if all(
            normalize_text(v) == ""
            for v in padded
        ):
            continue

        row_dict = {
            headers[i]: padded[i]
            for i in range(
                len(headers)
            )
            if headers[i]
        }

        records.append({
            "sheet_row": actual_row,
            "restaurant_id": row_dict.get(
                "Restaurant ID",
                ""
            ),
            "name": normalize_text(
                row_dict.get(
                    "Restaurant Name",
                    ""
                )
            ),
            "city": normalize_text(
                row_dict.get(
                    "City",
                    ""
                )
            ),
            "area": normalize_text(
                row_dict.get(
                    "Area",
                    ""
                )
            ),
            "maps_url": normalize_text(
                row_dict.get(
                    "Google Maps Link",
                    ""
                )
            ),
            **row_dict,
        })

    return records


existing_records = (
    read_existing_master()
)

print(
    "Existing Master rows:",
    len(existing_records)
)


Existing Master rows: 0


In [ ]:

# CELL 17 — Drive checkpoint + log

def now_iso():
    return datetime.now(
        timezone.utc
    ).isoformat()


def load_checkpoint():

    if (
        RESET_CHECKPOINT
        or not os.path.exists(
            CHECKPOINT_JSON
        )
    ):

        return {
            "created_at": now_iso(),
            "updated_at": now_iso(),
            "records": {},
            "logs": [],
        }

    try:

        with open(
            CHECKPOINT_JSON,
            "r",
            encoding="utf-8"
        ) as f:

            state = json.load(f)

        print(
            "Loaded checkpoint records:",
            len(
                state.get(
                    "records",
                    {}
                )
            )
        )

        return state

    except Exception as exc:

        print(
            "Checkpoint load failed:",
            exc
        )

        return {
            "created_at": now_iso(),
            "updated_at": now_iso(),
            "records": {},
            "logs": [],
        }


checkpoint = load_checkpoint()


def save_checkpoint():

    checkpoint["updated_at"] = now_iso()

    temp_path = (
        CHECKPOINT_JSON
        + ".tmp"
    )

    with open(
        temp_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            checkpoint,
            f,
            ensure_ascii=False,
            indent=2,
            default=str
        )

    os.replace(
        temp_path,
        CHECKPOINT_JSON
    )


def checkpoint_key(
    mode,
    url
):

    return (
        f"{mode}|"
        f"{normalize_text(url).split('?')[0]}"
    )


def get_cached(
    mode,
    url
):

    return checkpoint[
        "records"
    ].get(
        checkpoint_key(
            mode,
            url
        )
    )


def save_record_checkpoint(
    mode,
    url,
    record
):

    checkpoint[
        "records"
    ][
        checkpoint_key(
            mode,
            url
        )
    ] = record

    save_checkpoint()


def append_log(
    entry
):

    checkpoint[
        "logs"
    ].append(
        entry
    )

    pd.DataFrame(
        checkpoint["logs"]
    ).to_csv(
        SCRAPE_LOG_CSV,
        index=False
    )

print("Checkpoint/log system loaded.")


Loaded checkpoint records: 20
Checkpoint/log system loaded.


In [ ]:

# CELL 18 — REFRESH mode

async def run_refresh(context):

    if not RUN_REFRESH:

        print(
            "REFRESH disabled."
        )

        return []

    page = await context.new_page()

    candidates = [
        r
        for r in existing_records
        if r.get("maps_url")
    ]

    if MAX_REFRESH_RECORDS is not None:

        candidates = candidates[
            :MAX_REFRESH_RECORDS
        ]

    total = len(
        candidates
    )

    print(
        f"Refreshing {total} records..."
    )

    results = []

    for idx, base in enumerate(
        candidates,
        start=1
    ):

        url = base["maps_url"]

        cached = get_cached(
            "REFRESH",
            url
        )

        if cached:

            cached["existing_sheet_row"] = (
                base["sheet_row"]
            )

            cached["existing_restaurant_id"] = (
                base.get(
                    "Restaurant ID",
                    ""
                )
            )

            results.append(
                cached
            )

            print(
                f"[{idx}/{total}] "
                f"RESUME {base['name']} | "
                f"Reviews={cached.get('review_count')}"
            )

            continue

        started = now_iso()

        record = await extract_place_details(
            page,
            url=url,
            city=base.get(
                "City",
                ""
            ),
            area=base.get(
                "Area",
                ""
            ),
            search_category="",
            mode="REFRESH"
        )

        record[
            "existing_sheet_row"
        ] = base[
            "sheet_row"
        ]

        record[
            "existing_restaurant_id"
        ] = base.get(
            "Restaurant ID",
            ""
        )

        results.append(
            record
        )

        save_record_checkpoint(
            "REFRESH",
            url,
            record
        )

        append_log({
            "timestamp": started,
            "mode": "REFRESH",
            "restaurant_id": base.get(
                "Restaurant ID",
                ""
            ),
            "restaurant": base.get(
                "name",
                ""
            ),
            "url": url,
            "place_id": record.get(
                "place_id",
                ""
            ),
            "rating": record.get(
                "rating"
            ),
            "review_count": record.get(
                "review_count"
            ),
            "review_source": record.get(
                "review_source"
            ),
            "hours": record.get(
                "hours",
                ""
            ),
            "dine_in": record.get(
                "dine_in",
                ""
            ),
            "takeaway": record.get(
                "takeaway",
                ""
            ),
            "delivery": record.get(
                "delivery",
                ""
            ),
            "current_open_status": record.get(
                "current_open_status",
                ""
            ),
            "business_status": record.get(
                "business_status",
                ""
            ),
            "data_confidence": record.get(
                "data_confidence",
                ""
            ),
            "status": (
                "ERROR"
                if record.get("error")
                else "OK"
            ),
            "error": record.get(
                "error",
                ""
            ),
        })

        pct = round(
            idx / total * 100,
            1
        )

        print(
            f"[{idx}/{total} — {pct}%] "
            f"{record.get('name')} | "
            f"Rating={record.get('rating')} | "
            f"Reviews={record.get('review_count')} | "
            f"PlaceID={bool(record.get('place_id'))}"
        )

        await asyncio.sleep(
            random.uniform(
                MIN_DELAY,
                MAX_DELAY
            )
        )

    await page.close()

    return results


In [ ]:

# CELL 19 — SEARCH mode

async def run_search(context):

    if not RUN_SEARCH:

        print(
            "SEARCH disabled."
        )

        return []

    page = await context.new_page()

    search_plan = queries

    if MAX_SEARCH_QUERIES is not None:

        search_plan = search_plan[
            :MAX_SEARCH_QUERIES
        ]

    results = []

    for q_idx, query_info in enumerate(
        search_plan,
        start=1
    ):

        print(
            f"\nQUERY "
            f"{q_idx}/{len(search_plan)}"
        )

        search_results = (
            await collect_search_results(
                page,
                query_info["query"],
                MAX_RESULTS_PER_QUERY
            )
        )

        for r_idx, result in enumerate(
            search_results,
            start=1
        ):

            url = result["url"]

            cached = get_cached(
                "SEARCH",
                url
            )

            if cached:

                cached["source_query"] = (
                    query_info["query"]
                )

                results.append(
                    cached
                )

                print(
                    f"  [{r_idx}/"
                    f"{len(search_results)}] "
                    f"RESUME "
                    f"{cached.get('name')} | "
                    f"Reviews="
                    f"{cached.get('review_count')}"
                )

                continue

            started = now_iso()

            # IMPORTANT:
            # SEARCH uses the exact same detail extractor
            # as REFRESH.
            record = (
                await extract_place_details(
                    page,
                    url=url,
                    city=query_info["city"],
                    area=query_info["area"],
                    search_category=query_info["category"],
                    mode="SEARCH"
                )
            )

            record[
                "source_query"
            ] = query_info[
                "query"
            ]

            results.append(
                record
            )

            save_record_checkpoint(
                "SEARCH",
                url,
                record
            )

            append_log({
                "timestamp": started,
                "mode": "SEARCH",
                "restaurant_id": "",
                "restaurant": record.get(
                    "name",
                    ""
                ),
                "url": url,
                "query": query_info["query"],
                "city": query_info["city"],
                "area": query_info["area"],
                "category": query_info["category"],
                "place_id": record.get(
                    "place_id",
                    ""
                ),
                "rating": record.get(
                    "rating"
                ),
                "review_count": record.get(
                    "review_count"
                ),
                "review_source": record.get(
                    "review_source"
                ),
                "hours": record.get(
                    "hours",
                    ""
                ),
                "dine_in": record.get(
                    "dine_in",
                    ""
                ),
                "takeaway": record.get(
                    "takeaway",
                    ""
                ),
                "delivery": record.get(
                    "delivery",
                    ""
                ),
                "current_open_status": record.get(
                    "current_open_status",
                    ""
                ),
                "business_status": record.get(
                    "business_status",
                    ""
                ),
                "data_confidence": record.get(
                    "data_confidence",
                    ""
                ),
                "status": (
                    "ERROR"
                    if record.get("error")
                    else "OK"
                ),
                "error": record.get(
                    "error",
                    ""
                ),
            })

            print(
                f"  [{r_idx}/"
                f"{len(search_results)}] "
                f"{record.get('name')} | "
                f"Rating="
                f"{record.get('rating')} | "
                f"Reviews="
                f"{record.get('review_count')} | "
                f"PlaceID="
                f"{bool(record.get('place_id'))}"
            )

            await asyncio.sleep(
                random.uniform(
                    MIN_DELAY,
                    MAX_DELAY
                )
            )

    await page.close()

    return results


In [ ]:

# CELL 20 — Run scraper

async def run_collection():

    async with async_playwright() as p:

        browser = await p.chromium.launch(
            headless=HEADLESS,
            args=[
                "--no-sandbox",
                "--disable-dev-shm-usage",
                "--disable-blink-features=AutomationControlled",
            ]
        )

        context = await browser.new_context(
            viewport=VIEWPORT,
            locale="en-US",
            user_agent=(
                "Mozilla/5.0 "
                "(Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 "
                "(KHTML, like Gecko) "
                "Chrome/125.0.0.0 "
                "Safari/537.36"
            ),
        )

        refresh_results = await run_refresh(
            context
        )

        search_results = await run_search(
            context
        )

        await browser.close()

    return (
        refresh_results,
        search_results
    )


refresh_results, search_results = (
    await run_collection()
)

print("\n==============================")
print("COLLECTION COMPLETE")
print("==============================")
print(
    "Refresh:",
    len(refresh_results)
)
print(
    "Search:",
    len(search_results)
)


Refreshing 0 records...

QUERY 1/2

SEARCH: restaurants in F-6 Islamabad
  scroll 01: 7 URLs
  scroll 02: 14 URLs
  [1/10] RESUME The Smokey Cauldron | Reviews=3717
  [2/10] RESUME MêZ Islamabad Turkish Restaurant | Reviews=1006
  [3/10] RESUME Mandarin Kitchen | Reviews=1263
  [4/10] RESUME The Carnivore | Reviews=8030
  [5/10] RESUME Santorini Steak House By Bread Chef Café & Bakers | Reviews=779
  [6/10] RESUME Bawa & Co - South Asian Fusion | Reviews=150
  [7/10] District 6 | Rating=4.1 | Reviews=1283 | PlaceID=True
  [8/10] RESUME Gai'a Japanese Fusion | Reviews=254
  [9/10] RESUME Rumba - The Meat Factory | Reviews=811
  [10/10] RESUME Tayto F6 | Reviews=1765

QUERY 2/2

SEARCH: cafes in F-6 Islamabad
  scroll 01: 7 URLs
  scroll 02: 14 URLs
  [1/10] RESUME Caffé Praha - Islamabad | Reviews=4653
  [2/10] RESUME KAF Coffee | Reviews=781
  [3/10] RESUME Roasters Coffee House & Grill | Reviews=1018
  [4/10] RESUME Street 1 Cafe | Reviews=2751
  [5/10] RESUME chaayé khana F-6 | Revie

In [ ]:

# CELL 21 — Deduplicate + diagnostics

def deduplicate_records(records):

    unique = {}
    duplicates = 0

    for record in records:

        key = make_identity_key(
            record
        )

        if key in unique:

            duplicates += 1

            existing = unique[key]

            # Fill missing fields only.
            for field, value in record.items():

                if (
                    existing.get(field)
                    in (None, "")
                    and value
                    not in (None, "")
                ):

                    existing[field] = value

            # Prefer a real review count.
            if (
                existing.get(
                    "review_count"
                ) is None
                and record.get(
                    "review_count"
                ) is not None
            ):

                existing[
                    "review_count"
                ] = record[
                    "review_count"
                ]

                existing[
                    "review_source"
                ] = record.get(
                    "review_source",
                    ""
                )

        else:

            unique[key] = dict(
                record
            )

    print(
        "Collected:",
        len(records)
    )

    print(
        "Duplicates removed:",
        duplicates
    )

    print(
        "Unique:",
        len(unique)
    )

    return list(
        unique.values()
    )


all_records = (
    refresh_results
    + search_results
)

unique_records = deduplicate_records(
    all_records
)

with open(
    RAW_RESULTS_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        unique_records,
        f,
        ensure_ascii=False,
        indent=2,
        default=str
    )


def quality_report(records):

    total = len(records)

    if not total:
        print(
            "No records."
        )
        return

    fields = [
        ("Rating", "rating"),
        ("Review Count", "review_count"),
        ("Place ID", "place_id"),
        ("Maps URL", "maps_url"),
        ("Phone", "phone"),
        ("Address", "address"),
        ("Website", "website"),
        ("Hours", "hours"),
        ("Dine-In", "dine_in"),
        ("Takeaway", "takeaway"),
        ("Delivery", "delivery"),
        ("Google Category", "category"),
    ]

    print(
        "\n========== QUALITY =========="
    )

    print(
        "Total records:",
        total
    )

    for title, field in fields:

        filled = sum(
            1
            for r in records
            if r.get(field)
            not in (None, "")
        )

        pct = round(
            filled / total * 100,
            1
        )

        print(
            f"{title:18s} "
            f"{filled:4d}/{total} "
            f"({pct}%)"
        )

    print(
        "\nReview sources:"
    )

    print(
        pd.Series([
            r.get(
                "review_source",
                "unknown"
            )
            for r in records
        ]).value_counts(
            dropna=False
        ).to_string()
    )

    print(
        "\nConfidence:"
    )

    print(
        pd.Series([
            r.get(
                "data_confidence",
                "UNKNOWN"
            )
            for r in records
        ]).value_counts(
            dropna=False
        ).to_string()
    )


quality_report(
    unique_records
)


Collected: 20
Duplicates removed: 0
Unique: 20

========== QUALITY ==========
Total records: 20
Rating               20/20 (100.0%)
Review Count         20/20 (100.0%)
Place ID             17/20 (85.0%)
Maps URL             20/20 (100.0%)
Phone                20/20 (100.0%)
Address              20/20 (100.0%)
Website              11/20 (55.0%)
Hours                18/20 (90.0%)
Dine-In               0/20 (0.0%)
Takeaway             18/20 (90.0%)
Delivery              0/20 (0.0%)
Google Category      20/20 (100.0%)

Review sources:
aria-label    20

Confidence:
OK                          17
REVIEW: Place ID missing     3


In [ ]:

# CELL 22 — Diagnostics table BEFORE Google Sheet write

def diagnostics_table(records):

    rows = []

    for r in records:

        rows.append({
            "Mode": r.get(
                "mode",
                ""
            ),
            "Restaurant": r.get(
                "name",
                ""
            ),
            "Rating": r.get(
                "rating"
            ),
            "Reviews": r.get(
                "review_count"
            ),
            "Review Source": r.get(
                "review_source",
                ""
            ),
            "Place ID": r.get(
                "place_id",
                ""
            ),
            "Maps Internal ID": r.get(
                "maps_internal_id",
                ""
            ),
            "Hours": r.get(
                "hours",
                ""
            ),
            "Dine-In": r.get(
                "dine_in",
                ""
            ),
            "Takeaway": r.get(
                "takeaway",
                ""
            ),
            "Delivery": r.get(
                "delivery",
                ""
            ),
            "Current Open": r.get(
                "current_open_status",
                ""
            ),
            "Business Status": r.get(
                "business_status",
                ""
            ),
            "Confidence": r.get(
                "data_confidence",
                ""
            ),
            "Error": r.get(
                "error",
                ""
            ),
        })

    diag = pd.DataFrame(
        rows
    )

    display(diag)

    return diag


diagnostics_df = diagnostics_table(
    unique_records
)


,Mode,Restaurant,Rating,Reviews,Review Source,Place ID,Maps Internal ID,Hours,Dine-In,Takeaway,Delivery,Current Open,Business Status,Confidence,Error
0,SEARCH,The Smokey Cauldron,4.3,3717,aria-label,ChIJ3z2wg-Ts3zgR5y_bF5bv_LA,0x38dfece483b03ddf:0xb0fcef9617db2fe7,Mon–Sun: 11:30 AM–12 AM,,Y,,Open,Operational,OK,
1,SEARCH,MêZ Islamabad Turkish Restaurant,4.2,1006,aria-label,ChIJ6U-XQPq_3zgRomTR-4SFYKY,0x38dfbffa40974fe9:0xa6608584fbd164a2,Mon–Sun: 1 PM–12 AM,,Y,,Open,Operational,OK,
2,SEARCH,Mandarin Kitchen,4.2,1263,aria-label,ChIJV0u3YI6_3zgRWtzSLVIGCbQ,0x38dfbf8e60b74b57:0xb40906522dd2dc5a,Mon–Sun: 11 PM,,Y,,Open,Operational,OK,
3,SEARCH,The Carnivore,4.7,8030,aria-label,,0x38dfbf448ddd3747:0x174f73aa9c9943a1,Mon–Sun: 11 PM,,Y,,Open,Operational,REVIEW: Place ID missing,
4,SEARCH,Santorini Steak House By Bread Chef Café & Bakers,4.6,779,aria-label,ChIJ6RL7YAC_3zgRr-l_jfISnLs,0x38dfbf0060fb12e9:0xbb9c12f28d7fe9af,Mon–Sun: 7 AM–11 PM,,Y,,Open,Operational,OK,
5,SEARCH,Bawa & Co - South Asian Fusion,4.9,150,aria-label,ChIJp4a_6R2_3zgRSq38m3STafA,0x38dfbf1de9bf86a7:0xf06993749bfcad4a,Mon–Sun: 11 PM,,Y,,Open,Operational,OK,
6,SEARCH,District 6,4.1,1283,aria-label,ChIJ7SNT6nm_3zgRK13ekVO5LEg,0x38dfbf79ea5323ed:0x482cb95391de5d2b,Mon–Fri: 11 PM | Sat–Sun: 10:30 AM–11 PM,,Y,,Open,Operational,OK,
7,SEARCH,Gai'a Japanese Fusion,4.3,254,aria-label,ChIJ9wUw89-_3zgRhq0SkDEYb_Q,0x38dfbfdff33005f7:0xf46f18319012ad86,Mon–Sun: 11:45 PM,,Y,,Open,Operational,OK,
8,SEARCH,Rumba - The Meat Factory,4.3,811,aria-label,ChIJ9yu6jGm_3zgRydUErxVsxzM,0x38dfbf698cba2bf7:0x33c76c15af04d5c9,,,Y,,Open,Operational,OK,
9,SEARCH,Tayto F6,4.5,1765,aria-label,ChIJ2RaGVCC_3zgRnq5HES27W0A,0x38dfbf20548616d9:0x405bbb2d1147ae9e,,,Y,,Open,Operational,OK,


In [ ]:

# CELL 23 — Google Sheets writer helpers

def current_master_state():

    ws = spreadsheet.worksheet(
        "Restaurant Master"
    )

    values = ws.get_all_values()

    if not values:
        raise ValueError(
            "Restaurant Master is empty."
        )

    headers = [
        normalize_text(
            x
        )
        for x in values[0]
    ]

    return ws, values, headers


def ensure_v231_columns_gspread(
    ws,
    headers
):

    headers = list(
        headers
    )

    added = []

    for column in V231_COLUMNS:

        if column not in headers:

            headers.append(
                column
            )

            added.append(
                column
            )

    if added:

        # Add headers at the end.
        start_col = (
            len(headers)
            - len(added)
            + 1
        )

        for offset, name in enumerate(
            added
        ):

            ws.update_cell(
                1,
                start_col + offset,
                name
            )

    return {
        h: i
        for i, h in enumerate(
            headers,
            start=1
        )
        if h
    }


def normalize_maps_url(url):

    return (
        normalize_text(
            url
        )
        .split("?")[0]
        .rstrip("/")
    )


def build_existing_identity_maps(
    ws,
    values,
    headers
):

    maps = {
        "place_id": {},
        "maps": {},
        "phone": {},
        "name_address": {},
    }

    for row_number, raw_row in enumerate(
        values[1:],
        start=2
    ):

        padded = list(raw_row) + [
            ""
        ] * (
            len(headers)
            - len(raw_row)
        )

        row = {
            headers[i]: padded[i]
            for i in range(
                len(headers)
            )
            if headers[i]
        }

        place_id = normalize_text(
            row.get(
                "Place ID",
                ""
            )
        )

        if place_id:

            maps[
                "place_id"
            ][place_id] = row_number

        maps_url = normalize_maps_url(
            row.get(
                "Google Maps Link",
                ""
            )
        )

        if maps_url:

            maps[
                "maps"
            ][maps_url] = row_number

        phone = phone_key(
            row.get(
                "Phone Number",
                ""
            )
        )

        if phone:

            maps[
                "phone"
            ][phone] = row_number

        name = normalize_name(
            row.get(
                "Restaurant Name",
                ""
            )
        )

        address = normalize_name(
            row.get(
                "Full Address",
                ""
            )
        )

        if name and address:

            maps[
                "name_address"
            ][
                f"{name}|{address}"
            ] = row_number

    return maps


def next_restaurant_id_from_values(
    values,
    headers
):

    if "Restaurant ID" not in headers:
        return 1

    idx = headers.index(
        "Restaurant ID"
    )

    ids = []

    for raw_row in values[1:]:

        if idx >= len(raw_row):
            continue

        value = raw_row[idx]

        try:
            ids.append(
                int(
                    float(
                        value
                    )
                )
            )
        except Exception:
            pass

    return (
        max(ids) + 1
        if ids
        else 1
    )


def build_updates_for_record(
    record
):

    updates = {}

    mappings = [
        ("name", "Restaurant Name"),
        ("address", "Full Address"),
        ("phone", "Phone Number"),
        ("website", "Website URL"),
        ("city", "City"),
        ("area", "Area"),
    ]

    for source, target in mappings:

        value = record.get(
            source
        )

        if value not in (
            None,
            ""
        ):

            updates[
                target
            ] = value

    if record.get(
        "rough_brand"
    ):
        updates[
            "Brand Name"
        ] = record[
            "rough_brand"
        ]

    if record.get(
        "format"
    ):
        updates[
            "Format"
        ] = record[
            "format"
        ]

    if record.get(
        "category"
    ):
        updates[
            "Cuisine Types"
        ] = record[
            "category"
        ]

    if record.get(
        "latitude"
    ) is not None:

        updates[
            "Latitude"
        ] = record[
            "latitude"
        ]

    if record.get(
        "longitude"
    ) is not None:

        updates[
            "Longitude"
        ] = record[
            "longitude"
        ]

    if record.get(
        "hours"
    ):

        updates[
            "Operational Timing"
        ] = record[
            "hours"
        ]

    if record.get(
        "website_type"
    ):

        updates[
            "Website Type"
        ] = record[
            "website_type"
        ]

    for source, target in [
        ("dine_in", "Dine-In (Y/N)"),
        ("takeaway", "Takeaway (Y/N)"),
        ("delivery", "Delivery (Y/N)"),
    ]:

        if record.get(source) == "Y":

            updates[
                target
            ] = "Y"

    updates[
        "Google Business (Y/N)"
    ] = "Y"

    if record.get(
        "maps_url"
    ):

        updates[
            "Google Maps Link"
        ] = record[
            "maps_url"
        ]

    if record.get(
        "rating"
    ) is not None:

        updates[
            "Google Star Rating"
        ] = record[
            "rating"
        ]

    if record.get(
        "review_count"
    ) is not None:

        updates[
            "Google Review Count"
        ] = record[
            "review_count"
        ]

    updates[
        "Data Source"
    ] = "Google Maps"

    updates[
        "Date Profiled"
    ] = datetime.now().strftime(
        "%Y-%m-%d"
    )

    if record.get(
        "foodpanda_listed"
    ) == "Y":

        updates[
            "Foodpanda Listed (Y/N)"
        ] = "Y"

    if record.get(
        "foodpanda_url"
    ):

        updates[
            "Foodpanda URL"
        ] = record[
            "foodpanda_url"
        ]

    notes = []

    if record.get(
        "current_open_status"
    ):
        notes.append(
            "CurrentOpen="
            + str(
                record[
                    "current_open_status"
                ]
            )
        )

    if record.get(
        "business_status"
    ):
        notes.append(
            "BusinessStatus="
            + str(
                record[
                    "business_status"
                ]
            )
        )

    if record.get(
        "price_level"
    ):
        notes.append(
            "Price="
            + str(
                record[
                    "price_level"
                ]
            )
        )

    if record.get(
        "review_source"
    ):
        notes.append(
            "ReviewSource="
            + str(
                record[
                    "review_source"
                ]
            )
        )

    if record.get(
        "maps_internal_id"
    ):
        notes.append(
            "MapsInternalID="
            + str(
                record[
                    "maps_internal_id"
                ]
            )
        )

    if record.get(
        "source_query"
    ):
        notes.append(
            "Query="
            + str(
                record[
                    "source_query"
                ]
            )
        )

    if notes:
        updates[
            "Notes"
        ] = " | ".join(
            notes
        )

    return updates


In [ ]:
def locate_record_row(
    record,
    identity_maps
):

    place_id = normalize_text(
        record.get(
            "place_id"
        )
    )

    # Primary identity.
    if place_id:

        return identity_maps[
            "place_id"
        ].get(
            place_id
        )

    maps_url = normalize_maps_url(
        record.get(
            "maps_url"
        )
    )

    if maps_url:

        found = identity_maps[
            "maps"
        ].get(
            maps_url
        )

        if found:
            return found

    phone = phone_key(
        record.get(
            "phone"
        )
    )

    if phone:

        found = identity_maps[
            "phone"
        ].get(
            phone
        )

        if found:
            return found

    name = normalize_name(
        record.get(
            "name"
        )
    )

    address = normalize_name(
        record.get(
            "address"
        )
    )

    if name and address:

        return identity_maps[
            "name_address"
        ].get(
            f"{name}|{address}"
        )

    return None


def write_to_google_sheet(
    records
):

    ws, values, headers = (
        current_master_state()
    )

    header_map = (
        ensure_v231_columns_gspread(
            ws,
            headers
        )
    )

    # Refresh values/headers after adding columns.
    values = ws.get_all_values()
    headers = [
        normalize_text(x)
        for x in values[0]
    ]

    header_map = {
        h: i
        for i, h in enumerate(
            headers,
            start=1
        )
        if h
    }

    identity_maps = (
        build_existing_identity_maps(
            ws,
            values,
            headers
        )
    )

    next_id = (
        next_restaurant_id_from_values(
            values,
            headers
        )
    )

    # Last real data row.
    last_data_row = 1

    for row_number, raw_row in enumerate(
        values[1:],
        start=2
    ):

        if any(
            normalize_text(v)
            != ""
            for v in raw_row
        ):
            last_data_row = row_number

    # Update dict prevents duplicate
    # coordinates inside one API call.
    cell_updates = {}

    updated_count = 0
    added_count = 0
    id_assigned_count = 0

    for record in records:

        target_row = locate_record_row(
            record,
            identity_maps
        )

        is_new = (
            target_row is None
        )

        if is_new:

            last_data_row += 1

            target_row = (
                last_data_row
            )

            added_count += 1

            # Reserve the new row's identity.
            new_id = str(
                next_id
            )

            next_id += 1

            id_assigned_count += 1

            # Update identity maps immediately.
            place_id = normalize_text(
                record.get(
                    "place_id"
                )
            )

            if place_id:

                identity_maps[
                    "place_id"
                ][
                    place_id
                ] = target_row

            maps_url = normalize_maps_url(
                record.get(
                    "maps_url"
                )
            )

            if maps_url:

                identity_maps[
                    "maps"
                ][
                    maps_url
                ] = target_row

            phone = phone_key(
                record.get(
                    "phone"
                )
            )

            if phone:

                identity_maps[
                    "phone"
                ][
                    phone
                ] = target_row

            name = normalize_name(
                record.get(
                    "name"
                )
            )

            address = normalize_name(
                record.get(
                    "address"
                )
            )

            if name and address:

                identity_maps[
                    "name_address"
                ][
                    f"{name}|{address}"
                ] = target_row

        else:

            updated_count += 1

        # ---- Restaurant ID ----
        id_col = header_map.get(
            "Restaurant ID"
        )

        if id_col:

            current_id = ""

            if (
                target_row - 2
                < len(values) - 1
            ):

                try:
                    idx = (
                        target_row - 2
                    )

                    if idx >= 0:
                        raw = values[
                            idx + 1
                        ]

                        if (
                            headers.index(
                                "Restaurant ID"
                            )
                            < len(raw)
                        ):

                            current_id = (
                                raw[
                                    headers.index(
                                        "Restaurant ID"
                                    )
                                ]
                            )

                except Exception:
                    current_id = ""

            if is_new:

                cell_updates[
                    (
                        target_row,
                        id_col
                    )
                ] = str(
                    next_id - 1
                )

            elif normalize_text(
                current_id
            ) == "":

                cell_updates[
                    (
                        target_row,
                        id_col
                    )
                ] = str(
                    next_id
                )

                next_id += 1
                id_assigned_count += 1

        # ---- Business fields ----
        updates = (
            build_updates_for_record(
                record
            )
        )

        # V2.3.1 columns.
        updates[
            "Place ID"
        ] = normalize_text(
            record.get(
                "place_id",
                ""
            )
        )

        updates[
            "Google Category"
        ] = normalize_text(
            record.get(
                "category",
                ""
            )
        )

        updates[
            "Current Open Status"
        ] = normalize_text(
            record.get(
                "current_open_status",
                "Unknown"
            )
        )

        updates[
            "Business Status"
        ] = normalize_text(
            record.get(
                "business_status",
                "Unknown"
            )
        )

        updates[
            "Price Level"
        ] = normalize_text(
            record.get(
                "price_level",
                ""
            )
        )

        updates[
            "Data Confidence"
        ] = normalize_text(
            record.get(
                "data_confidence",
                "REVIEW: not evaluated"
            )
        )

        updates[
            "Instagram URL"
        ] = normalize_text(
            record.get(
                "instagram_url",
                ""
            )
        )

        updates[
            "Facebook URL"
        ] = normalize_text(
            record.get(
                "facebook_url",
                ""
            )
        )

        # New row default CRM status.
        if is_new:

            updates[
                "Status"
            ] = "New Lead"

        for column, value in (
            updates.items()
        ):

            if column not in header_map:
                continue

            if value in (
                None,
                ""
            ):
                continue

            cell_updates[
                (
                    target_row,
                    header_map[
                        column
                    ]
                )
            ] = value

    # Write one batch.
    if cell_updates:

        cells = [
            gspread.cell.Cell(
                row=row,
                col=col,
                value=value
            )
            for (
                row,
                col
            ), value
            in cell_updates.items()
        ]

        ws.update_cells(
            cells
        )

    print(
        "========================================"
    )

    print(
        "Google Sheet sync complete."
    )

    print(
        "Existing rows updated:",
        updated_count
    )

    print(
        "New rows added:",
        added_count
    )

    print(
        "Restaurant IDs assigned:",
        id_assigned_count
    )

    print(
        "========================================"
    )

    return ws


def ensure_v231_columns_gspread(
    ws,
    headers
):
    # Make a copy of the initial headers to avoid modifying it during iteration for calculation purposes.
    initial_headers = list(headers)
    added = []

    for column in V231_COLUMNS:
        # Check against initial_headers to see if column is truly new
        if column not in initial_headers:
            added.append(column)

    if added:
        current_cols = len(initial_headers)
        # Increased limit from 40 to 100 to allow extending the sheet with missing V2.3.1 columns
        max_cols = 100

        if current_cols + len(added) > max_cols:
            missing_but_cannot_add = [
                col for col in added
                if col not in initial_headers
            ]
            raise RuntimeError(
                f"Cannot add V2.3.1 columns: {', '.join(missing_but_cannot_add)}. "
                f"The sheet already has {current_cols} columns and the maximum allowed is {max_cols}. "
                f"Please ensure the required V2.3.1 columns are present within the first {max_cols} columns "
                f"or expand your Google Sheet to accommodate more columns."
            )

        # If we reach here, it means we have space or no new columns to add.
        # Now, actually add the columns to the sheet.
        start_col = len(initial_headers) + 1 # Add after the last existing column

        for offset, name in enumerate(added):
            ws.update_cell(
                1,
                start_col + offset,
                name
            )
            initial_headers.append(name) # Keep this list updated for the header_map build

    return {
        h: i
        for i, h in enumerate(
            initial_headers, # Use the potentially updated list of headers for the map
            start=1
        )
        if h
    }


write_to_google_sheet(
    unique_records
)

RuntimeError: Cannot add V2.3.1 columns: Google Category, Current Open Status, Business Status, Price Level, Instagram URL, Facebook URL. The sheet already has 40 columns and the maximum allowed is 40. Please ensure the required V2.3.1 columns are present within the first 40 columns or expand your Google Sheet to accommodate more columns.

In [ ]:

# CELL 25 — Repair Dashboard directly in Google Sheets

def repair_dashboard_google_sheet():

    try:

        dash = spreadsheet.worksheet(
            "Dashboard"
        )

        master = spreadsheet.worksheet(
            "Restaurant Master"
        )

    except Exception as exc:

        print(
            "Dashboard/master sheet not available:",
            exc
        )

        return

    master_headers = [
        normalize_text(x)
        for x in master.row_values(
            1
        )
    ]

    column_map = {
        header: idx + 1
        for idx, header
        in enumerate(
            master_headers
        )
        if header
    }

    # Convert numeric column numbers to letters.
    def col_letter(number):
        result = ""

        while number:
            number, rem = divmod(
                number - 1,
                26
            )

            result = (
                chr(
                    65 + rem
                )
                + result
            )

        return result

    range_end = 10000

    fields = {
        "Restaurant Name": "Restaurant Name",
        "City": "City",
        "POS Installed": "POS Installed (Y/N)",
        "No BI Team": "Dedicated BI Team (Y/N)",
        "Foodpanda Listed": "Foodpanda Listed (Y/N)",
    }

    if not all(
        field in column_map
        for field in fields.values()
    ):
        print(
            "Required master columns for dashboard "
            "repair are not all present."
        )

        return

    formulas = {
        "Total Restaurants": (
            f"=COUNTA('Restaurant Master'!"
            f"{col_letter(column_map['Restaurant Name'])}"
            f"2:{col_letter(column_map['Restaurant Name'])}"
            f"{range_end})"
        ),

        "Islamabad Count": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['City'])}"
            f"2:{col_letter(column_map['City'])}"
            f"{range_end},\"Islamabad\")"
        ),

        "Rawalpindi Count": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['City'])}"
            f"2:{col_letter(column_map['City'])}"
            f"{range_end},\"Rawalpindi\")"
        ),

        "POS Installed": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['POS Installed (Y/N)'])}"
            f"2:{col_letter(column_map['POS Installed (Y/N)'])}"
            f"{range_end},\"Y\")"
        ),

        "No BI Team": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['Dedicated BI Team (Y/N)'])}"
            f"2:{col_letter(column_map['Dedicated BI Team (Y/N)'])}"
            f"{range_end},\"N\")"
        ),

        "Foodpanda Listed": (
            f'=COUNTIF(\'Restaurant Master\'!'
            f"{col_letter(column_map['Foodpanda Listed (Y/N)'])}"
            f"2:{col_letter(column_map['Foodpanda Listed (Y/N)'])}"
            f"{range_end},\"Y\")"
        ),
    }

    dashboard_values = (
        dash.get_all_values()
    )

    updates = {}

    for row_number, row in enumerate(
        dashboard_values,
        start=1
    ):

        for col_number, value in enumerate(
            row,
            start=1
        ):

            label = normalize_text(
                value
            ).lower()

            for metric, formula in formulas.items():

                if (
                    metric.lower()
                    in label
                ):

                    # Put formula in the cell immediately
                    # to the right of the metric label.
                    updates[
                        (
                            row_number,
                            col_number + 1
                        )
                    ] = formula

    if updates:

        cells = [
            gspread.cell.Cell(
                row=row,
                col=col,
                value=value
            )
            for (
                row,
                col
            ), value in updates.items()
        ]

        dash.update_cells(
            cells
        )

        print(
            "Dashboard formulas repaired:",
            len(cells)
        )

    else:

        print(
            "No matching Dashboard metric labels found."
        )


repair_dashboard_google_sheet()


In [ ]:

# CELL 26 — Final Google Sheet verification

def verify_google_sheet():

    ws = spreadsheet.worksheet(
        "Restaurant Master"
    )

    values = ws.get_all_values()

    if not values:
        print(
            "Restaurant Master is empty."
        )
        return

    headers = [
        normalize_text(x)
        for x in values[0]
    ]

    print(
        "\n========== FINAL SHEET QA =========="
    )

    print(
        "Rows:",
        max(0, len(values) - 1)
    )

    fields = [
        "Restaurant ID",
        "Restaurant Name",
        "Google Maps Link",
        "Google Star Rating",
        "Google Review Count",
        "Place ID",
        "Google Category",
        "Current Open Status",
        "Business Status",
        "Data Confidence",
        "Operational Timing",
        "Dine-In (Y/N)",
        "Takeaway (Y/N)",
        "Delivery (Y/N)",
        "Website URL",
        "Website Type",
    ]

    for field in fields:

        if field not in headers:

            print(
                field + ": COLUMN MISSING"
            )

            continue

        index = headers.index(
            field
        )

        filled = 0

        for row in values[1:]:

            if (
                index < len(row)
                and normalize_text(
                    row[index]
                ) != ""
            ):

                filled += 1

        print(
            f"{field:25s}: "
            f"{filled}/{len(values) - 1}"
        )

    # Duplicate checks.
    def nonblank_column(
        field
    ):

        if field not in headers:
            return []

        index = headers.index(
            field
        )

        return [
            normalize_text(
                row[index]
            )
            for row in values[1:]
            if (
                index < len(row)
                and normalize_text(
                    row[index]
                )
            )
        ]

    for field in [
        "Restaurant ID",
        "Place ID",
        "Google Maps Link",
    ]:

        data = nonblank_column(
            field
        )

        duplicates = (
            len(data)
            - len(set(data))
        )

        print(
            f"Duplicate {field}:",
            duplicates
        )


verify_google_sheet()


In [ ]:

# CELL 27 — Optional export of audit files already saved on Drive

print(
    "Drive audit files:"
)

for path in [
    CHECKPOINT_JSON,
    SCRAPE_LOG_CSV,
    RAW_RESULTS_JSON,
]:

    exists = os.path.exists(
        path
    )

    print(
        " -",
        path,
        "✅" if exists else "❌"
    )

print(
    "\nGoogle Sheet remains the system of record."
)
